# 01 · Generating a catalog & market with the LLM world generator

The simulator runs on **setup directories** (notebook 00). Two of the five blocks in a
setup — the product `catalog.csv` and the `market:` block of `setup.yaml` — are pure
*domain data*: what products exist, how they're priced, and how demand swings seasonally.
That data is tedious to author by hand and easy to get internally inconsistent.

`src/llm/` ships a generator that drafts both from a single **archetype** string
(`"fashion_retail"`, `"grocery"`, `"sports_cars"`, …):

| it writes | it does **not** write |
|---|---|
| `catalog.csv` (one row per SKU) | `nodes:` / `edges:` (topology) |
| the `market:` block of `setup.yaml` | `policy:` blocks |
|  | `run:` / `disruption:` |

Topology, policies, disruption and run parameters stay the modeller's job — exactly the
split from notebook 00. The setup directory also doubles as a **cache**: a repeat
`build_setup(...)` call that finds an existing `catalog.csv` returns it without touching
the LLM.

> This notebook makes a **real, paid LLM call** and therefore **requires an
> `OPENAI_API_KEY`** (via the `api_key=` kwarg, the environment, or a repo-root `.env`).
> It generates a tiny catalog live, inspects exactly what was written, then scaffolds that
> output into a runnable scenario.

In [1]:
%matplotlib inline
import os
from pathlib import Path

# Hop up to the repo root so `import src...` and relative paths resolve
# no matter where the kernel started.
while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 90

DEMO = Path("data/notebook_demos/01")   # all outputs land here (gitignored)
DEMO.mkdir(parents=True, exist_ok=True)
# Where the live generation writes (and where the optional reload reads from).
LIVE_DIR = DEMO / "fashion_retail_live"
print("cwd:", Path.cwd())

cwd: /Users/mislavjordanic/Documents/projects/personal_projects/supply_chain_simulator


## 1 · The API, explained

The whole generator is two classes:

- `OpenAIClient()` — wraps OpenAI's structured-output endpoint. Its constructor loads a
  repo-root `.env`, then resolves an `OPENAI_API_KEY` (from the `api_key=` kwarg, the
  environment, or that `.env`) and **raises `ValueError` if no key is found**.
- `WorldBuilder(archetype, client)` — the pipeline orchestrator. The one entry point is:

  ```python
  catalog, market = WorldBuilder("fashion_retail", OpenAIClient()).build_setup(
      n_items=50, setup_dir="setups/fashion_retail",
  )
  ```

`build_setup(n_items, setup_dir, *, force_rebuild=False)` returns a `(catalog, market)`
tuple — a `list[Ware]` and a `MarketParams`. On a **cache miss** it runs the LLM pipeline
(market slice → taxonomy → catalog naming → chunked correlations → chunked freshness; total
`2 + ceil(n/50) + ceil(n/50)` calls) and writes `catalog.csv` + the `market:` block. On a
**cache hit** (an existing `setup_dir/catalog.csv`) it loads from disk with zero LLM calls.

### The live call

This is a **real, paid** call. We construct the client — which loads `.env` and raises
`ValueError` if no key is resolvable — and generate a deliberately **tiny** `n_items` to
keep cost and latency trivial. There is no offline fallback: without a key this cell raises,
by design.

In [ ]:
from src.llm.world_builder import WorldBuilder
from src.llm.openai_client import OpenAIClient

# Real, paid call. OpenAIClient() loads .env and raises ValueError if no key
# is resolvable — this notebook requires a key (no offline fallback). To avoid
# paying on a re-run, see the optional "reuse a previous run" cell below.
#
# Model selection — keep cost down by optionally using a cheaper model:
#   • Default: leave MODEL = None to use OpenAIClient's built-in default
#     ("gpt-5.4-mini"; see src/llm/README.md).
#   • Cheaper / other: set the model name explicitly to any model id your key
#     can access (e.g. a smaller sibling such as "gpt-5.4-nano"), either in code:
#         MODEL = "gpt-5.4-nano"
#     or — without editing the notebook — via the environment or repo-root .env:
#         OPENAI_MODEL=gpt-5.4-nano
#     The env var is read below, so it wins when set; otherwise MODEL applies.
MODEL = os.getenv("OPENAI_MODEL") or None   # None ⇒ OpenAIClient's default model

# Pass model= only when chosen, so a None falls back to the client's default.
client = OpenAIClient(model=MODEL) if MODEL else OpenAIClient()
builder = WorldBuilder("fashion_retail", client)
# Small n_items keeps this cheap and fast. LIVE_DIR is defined in the setup cell.
live_catalog, live_market = builder.build_setup(
    n_items=8, setup_dir=str(LIVE_DIR),
)
print(f"live call ok — {len(live_catalog)} wares generated into {LIVE_DIR} "
      f"(model={client.model})")

### Optional · reuse a previous run (skip the paid call)

The live call cached its output into `LIVE_DIR` — `catalog.csv` and the `market:` block of
`setup.yaml`. If you've already generated once (this kernel or a previous session), you can
repopulate `live_catalog` / `live_market` straight from those files instead of paying for
another generation. Set `LOAD_FROM_DISK = True` and run this cell **in place of** the live
call above — it needs no API key. Leave it `False` to use whatever the live call produced.

In [2]:
# OPTIONAL — reuse a previous run instead of paying for the live call above.
# build_setup wrote catalog.csv + the market: block of setup.yaml into LIVE_DIR.
# Flip this to True and run this cell *in place of* the live call to repopulate
# live_catalog / live_market straight from those files — no API key, no cost.
LOAD_FROM_DISK = True

if LOAD_FROM_DISK:
    import yaml
    from src.sim.setup_io import _parse_catalog, _parse_market

    if not (LIVE_DIR / "catalog.csv").exists():
        raise FileNotFoundError(
            f"no cached output at {LIVE_DIR} — run the live call above once first")

    live_catalog = _parse_catalog(LIVE_DIR / "catalog.csv")
    market_raw = yaml.safe_load((LIVE_DIR / "setup.yaml").read_text())["market"]
    live_market = _parse_market(market_raw, "setup.yaml.market")
    print(f"loaded {len(live_catalog)} wares + market from {LIVE_DIR} (no LLM call)")
else:
    print("LOAD_FROM_DISK is False — using the live-call output from the cell above")

loaded 8 wares + market from data/notebook_demos/01/fashion_retail_live (no LLM call)


## 2 · What the generator wrote

`build_setup` persisted two things into `LIVE_DIR`: a `catalog.csv` (one row per SKU) and
the `market:` block of `setup.yaml`. It writes **only** those two blocks — no nodes, edges,
or policies. Here's the live output: the catalog as a DataFrame, then the raw files on disk.

In [3]:
# Inspect exactly what the LLM generated and build_setup persisted.
print("catalog wares:", len(live_catalog))
print("market regions:", live_market.regions, "| cycle_len:", live_market.cycle_len)
display(pd.DataFrame(live_catalog)[["name", "category", "base_price", "unit_cost", "seasonality"]])

print("\n--- catalog.csv ---")
print((LIVE_DIR / "catalog.csv").read_text())
print("--- setup.yaml (market: block only) ---")
print((LIVE_DIR / "setup.yaml").read_text())

catalog wares: 8
market regions: ['NA', 'EU', 'UK', 'APAC', 'LATAM', 'MEAA'] | cycle_len: 365


,name,category,base_price,unit_cost,seasonality
0,Satin Rose Wrap Dress,Women’s Apparel,78.0,42.0,spring/summer
1,Ribbed Sage Lounge Set,Women’s Apparel,64.0,34.0,all_season
2,Cropped Denim Shacket 'Indigo Drift',Women’s Apparel,74.0,45.0,fall
3,Performance Oxford Button-Down 'City Grid',Men’s Apparel,58.0,30.0,spring
4,Velvet-Snap School Hoodie 'Mini Nova',Kids & Baby Clothing,36.0,18.0,fall/winter
5,CloudStep Running Sneakers 'AeroMint',Shoes & Footwear,92.0,54.0,all_season
6,Gilded Arc Mini Crossbody Bag,Accessories & Bags,68.0,36.0,summer
7,Seamless Sculpt Sports Bra 'VibeContour',Activewear & Athleisure,44.0,22.0,all_season



--- catalog.csv ---
product_id,name,category,base_price,unit_cost,seasonality,related_products,init_stock_share
P0000,Satin Rose Wrap Dress,Women’s Apparel,78.0,42.0,spring/summer,Gilded Arc Mini Crossbody Bag:0.62;CloudStep Running Sneakers 'AeroMint':0.38;Seamless Sculpt Sports Bra 'VibeContour':0.31,1.0
P0001,Ribbed Sage Lounge Set,Women’s Apparel,64.0,34.0,all_season,Seamless Sculpt Sports Bra 'VibeContour':0.57;Gilded Arc Mini Crossbody Bag:0.39;Velvet-Snap School Hoodie 'Mini Nova':0.33,1.0
P0002,Cropped Denim Shacket 'Indigo Drift',Women’s Apparel,74.0,45.0,fall,Satin Rose Wrap Dress:0.44;Gilded Arc Mini Crossbody Bag:0.49;Ribbed Sage Lounge Set:0.36,1.0
P0003,Performance Oxford Button-Down 'City Grid',Men’s Apparel,58.0,30.0,spring,Gilded Arc Mini Crossbody Bag:0.46;CloudStep Running Sneakers 'AeroMint':0.41;Cropped Denim Shacket 'Indigo Drift':0.34,1.0
P0004,Velvet-Snap School Hoodie 'Mini Nova',Kids & Baby Clothing,36.0,18.0,fall/winter,CloudStep Running Sneakers 'AeroMint':

## 3 · From generated data → a runnable setup

Generated data alone is **not runnable**: a setup with only a catalog and market has no
nodes, so nothing produces, orders, or buys. Two steps turn it into a live scenario:

1. **scaffold a topology** onto the catalog with `scaffold_topology` + `ScaffoldSpec`
   (notebook 00, route A) — one factory per product, a shared shop, one sink per
   (product × shop);
2. **attach policies** — a scaffolded topology has *no* `policy:` blocks, so it would run
   degenerate (no production / ordering / demand). We mirror the policy blocks from
   `setups/three_node_chain/setup.yaml` (`static_factory` on factories, `order_up_to` on
   shops, `default_demand_sink` on sinks) so the run produces real activity.

In [ ]:
import yaml
from src.sim.topology_scaffolder import scaffold_topology, ScaffoldSpec
from src.sim.setup_io import _parse_catalog

# Reload the LLM-generated catalog off disk so we scaffold off the real
# persisted data (identical to what build_setup wrote above).
gen_catalog = _parse_catalog(LIVE_DIR / "catalog.csv")

# Place nodes in a region the generated market actually defines. The LLM picks
# its own regions (e.g. "NA", "EU", …), so the ScaffoldSpec default of "US"
# would not exist in market_state and the run would KeyError. Use the market's
# first region so the topology and the market agree.
region = live_market.regions[0]

block = scaffold_topology(gen_catalog, ScaffoldSpec(shop_count=1, factory_capacity=120,
                                                    shop_capacity=400, factory_lead_time=2,
                                                    region=region))
print(f"scaffolded {len(block['nodes'])} nodes, {len(block['edges'])} edges in region {region!r}")
print("node types:", sorted({n['type'] for n in block['nodes']}))

### Attach policies to every scaffolded node

We walk the scaffolded `nodes:` and inject the per-type `policy:` block (and a few runtime
fields like a sink `demand_dist`) copied from the three-node-chain example. Then we hand-write
the `run:` and `disruption:` blocks and assemble the full `setup.yaml`.

In [ ]:
def attach_policies(nodes):
    """Mirror the policy blocks from setups/three_node_chain onto scaffolded nodes."""
    out = []
    for n in nodes:
        n = dict(n)
        if n["type"] == "factory":
            n["policy"] = {"name": "static_factory", "params": {"target_inventory": 200}}
            n["inventory"] = 100
        elif n["type"] == "intermediate":
            n["policy"] = {"name": "order_up_to",
                           "params": {"cover_horizon_ticks": 14, "list_price_out": 8.0}}
        elif n["type"] == "demand_sink":
            n["policy"] = {"name": "default_demand_sink", "params": {}}
            n.setdefault("demand_dist", {"kind": "normal", "mean": 10.0, "std": 2.0})
            n.setdefault("income_rate", 200.0)
        out.append(n)
    return out

doc = {
    "run": {"n_steps": 30, "start_date": "2024-01-01", "world_seed": 42},
    "market": yaml.safe_load((LIVE_DIR / "setup.yaml").read_text())["market"],
    "disruption": {"event_prob": 0.0, "types": ["natural_disaster"], "regions": [region],
                   "severity": {"kind": "constant", "value": 0.0},
                   "duration": {"kind": "constant", "value": 1}},
    "nodes": attach_policies(block["nodes"]),
    "edges": block["edges"],
}

run_dir = DEMO / "fashion_retail_runnable"
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "catalog.csv").write_text((LIVE_DIR / "catalog.csv").read_text())
(run_dir / "setup.yaml").write_text(yaml.safe_dump(doc, sort_keys=False))
print("wrote runnable setup:", sorted(p.name for p in run_dir.iterdir()))
print("nodes:", [(n["id"], n["type"]) for n in doc["nodes"]])

### Load it & smoke-run

`load_setup` validates and wires the policies; a 30-tick `Runner` confirms the generated
catalog + scaffolded topology actually transacts (cash and inventory move).

In [6]:
from src.sim.runner import Runner
from src.sim.inspect import node_timeseries_df
from src.sim.setup_io import load_setup

scenario = load_setup(run_dir)
display(scenario.summary_df().T)

run_log = Runner(scenario).run()
ts = node_timeseries_df(run_log, scenario)
final = ts.sort_values("tick").groupby(["node_type", "node_id"]).last()
display(final[["cash", "inventory_total"]])

moved = (final["cash"] != final["cash"].iloc[0]).any() or final["inventory_total"].sum() > 0
print(f"ran {run_log['n_steps']} ticks; activity observed:",
      bool(ts['inventory_total'].sum() > 0 or ts['orders_total'].sum() > 0))

,0
n_steps,30
start_date,2024-01-01 00:00:00
world_seed,42
n_products,8
n_nodes,17
n_DemandSinkNode,8
n_FactoryNode,8
n_IntermediateNode,1


KeyError: 'US'

## Where to next

| notebook | topic |
|---|---|
| **00** | setup directories — anatomy, inspection & the two authoring routes |
| **03** | topology gallery — build, draw, run & compare several DAG shapes |

The generator handles the *what to sell* half; notebooks 00 and 03 cover the *how it's
wired* half. Together they author a complete setup directory from a single archetype word.